<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/17_evaluacion_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 17. Evaluación en el test de Kingston

El test son **200 capturas tipo Google Maps de Kingston** (125 con paneles, 75 sin paneles, 178 cajas),
etiquetadas a mano con el criterio del train (`docs/criterios_datos.md`: una caja por grupo de paneles). Están en
la carpeta compartida, en `PDI-TP/test_kingston/` (ver su `LEEME.md`); en local se leen de
`resultados_locales/test_kingston/` con la misma estructura.

Este notebook toma los modelos ya guardados en Drive, predice el test y muestra:

1. las métricas de cada corrida en test, al lado de las de val, con intervalos bootstrap;
2. **ejemplos de cómo clasificó cada modelo y cómo era realmente** cada imagen (aciertos y errores, con las
   cajas anotadas y las predichas);
3. las imágenes en las que se equivocan casi todos los modelos.

**El test se usa una sola vez, al final.** Elegir modelos, umbrales o recetas mirando estos números convierte al
test en un segundo val y deja de medir de verdad: esas decisiones salen de val. Cada corrida usa el umbral que
eligió en val (0.5 las CNN, 0.25 los YOLO).

No entrena ni escribe nada: abre los resultados en solo lectura. Como el 13, corre **una familia por sesión**
(las CNN, YOLOv8n/YOLO11n y YOLO26n usan entornos distintos).

In [ ]:
FAMILIA = "clasificadores"  # "clasificadores" (CNN), "detectores" (YOLOv8n/YOLO11n) o "yolo26n": una sesión por familia

# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

assert FAMILIA in ("clasificadores", "detectores", "yolo26n"), FAMILIA
# YOLO26n necesita su propia Ultralytics: con la de requirements-detection.txt carga, pero predice otras cajas.
requisitos = {"clasificadores": "requirements-training.txt", "detectores": "requirements-detection.txt",
              "yolo26n": "requirements-yolo26.txt"}[FAMILIA]
if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / requisitos)], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

import hashlib
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

from pdi_tp import comparacion, datos, deteccion, entorno, resultados, split

rutas = entorno.preparar(solo_lectura=True)  # monta Drive, pero no deja crear ni guardar corridas
TEST = rutas.resultados_equipo.parent / "test_kingston"  # Drive: PDI-TP/test_kingston; local: resultados_locales/
if not (TEST / "etiquetas.csv").is_file():
    raise FileNotFoundError(f"No está el test en {TEST}: ver README (carpeta compartida test_kingston).")

## 1. El test

Se controla que las imágenes sean las etiquetadas (sha256 de `etiquetas.csv`) y que la etiqueta con/sin coincida
con el `.txt` de cajas: una imagen es "con paneles" si tiene al menos una caja.

In [ ]:
etiquetas = pd.read_csv(TEST / "etiquetas.csv")
imagenes = np.stack([datos.leer_imagen(TEST / "images" / nombre) for nombre in etiquetas["archivo"]])
reales = {nombre: deteccion.leer_cajas(TEST / "labels" / f"{Path(nombre).stem}.txt") for nombre in etiquetas["archivo"]}

distintas = [n for n, h in zip(etiquetas["archivo"], etiquetas["sha256"])
             if hashlib.sha256((TEST / "images" / n).read_bytes()).hexdigest() != h]
inconsistentes = [n for n, e in zip(etiquetas["archivo"], etiquetas["etiqueta"]) if (e == "con") != bool(len(reales[n]))]
if distintas or inconsistentes or imagenes.shape[1:] != (256, 256, 3):
    raise ValueError(f"Test modificado: imágenes distintas {distintas[:5]}, etiquetas inconsistentes {inconsistentes[:5]}")
y_test = (etiquetas["etiqueta"] == "con").astype(int).to_numpy()
pd.DataFrame({"imágenes": [len(y_test)], "con paneles": [y_test.sum()], "sin paneles": [(1 - y_test).sum()],
              "cajas": [sum(len(c) for c in reales.values())]}, index=["test"])

## 2. Predicciones de cada modelo

Entran las corridas **vigentes** de la familia elegida (las mismas que compara `15_comparacion_validacion`):

- **CNN:** el `.keras` de la corrida recibe la imagen RGB en 0–255; el preprocesado de cada red está dentro del
  modelo.
- **YOLO:** `modelo.pt` con los parámetros de su receta (`imgsz`, `conf`, `iou`, `max_det`). El puntaje de la imagen
  es la máxima confianza de sus cajas (0 si no hay), igual que en los notebooks 08 y 09. El mAP se calcula con
  `deteccion.map_cajas`, que reproduce `model.val` de Ultralytics sin escribir archivos; para eso las imágenes
  entran como en `model.val` (agrandadas a `imgsz` y con un borde gris de 16 px). Controlado contra `model.val`
  sobre este test: igual en YOLOv8n y YOLO11n (Ultralytics 8.3.221); en YOLO26n (8.4.184) difiere en menos de
  0.005.

**Control de la carga:** las predicciones de val tienen que coincidir con las guardadas en `predicciones_val.csv`.
Si la diferencia pasa de `TOLERANCIA`, la corrida no se está cargando igual que al entrenar y queda afuera.

In [ ]:
corridas = comparacion.corridas_vigentes(resultados.cargar_corridas(rutas))
familia = "clasificador" if FAMILIA == "clasificadores" else "detector"
de_familia = (corridas["familia"] == familia) & ((corridas["modelo"] == "yolo26n") == (FAMILIA == "yolo26n"))
no_vigentes = corridas.loc[de_familia & ~corridas["vigente"], "corrida"].tolist()
if no_vigentes:
    print("No vigentes, quedan afuera (otra receta):", no_vigentes)
corridas = corridas[de_familia & corridas["vigente"]].reset_index(drop=True)
if corridas.empty:
    raise RuntimeError(f"No hay corridas vigentes de {FAMILIA} en Drive.")
archivos_val = [p.name for p in split.cargar_split(rutas)["val"]]
imagenes_val = np.stack([datos.leer_imagen(rutas.imagenes / nombre) for nombre in archivos_val])
corridas[["corrida", "modelo", "modo", "fecha", "umbral"]]

In [ ]:
if FAMILIA == "clasificadores":
    from tensorflow import keras

    def predecir(carpeta, imagenes):
        # ResNet50 guarda el mejor checkpoint como mejor.keras; las demás CNN, como modelo.keras.
        pesos = next(carpeta / n for n in ("modelo.keras", "mejor.keras") if (carpeta / n).is_file())
        keras.backend.clear_session()
        modelo = keras.models.load_model(pesos, compile=False)
        return modelo.predict(imagenes.astype("float32"), batch_size=32, verbose=0).ravel(), None
else:
    import json
    from functools import partial

    import cv2
    import torch
    from ultralytics import YOLO
    from ultralytics.utils import nms

    # Sin GPU, el NMS con conf=0.001 puede pasar el límite de tiempo de Ultralytics y descartar cajas en silencio
    # (solo avisa con un warning). Se agranda el límite: en una T4 no cambia nada.
    nms.non_max_suppression = partial(nms.non_max_suppression, max_time_img=120)
    DISPOSITIVO = 0 if torch.cuda.is_available() else "cpu"

    def receta_de(carpeta):
        # La receta se lee de la corrida: no todos los YOLO guardados se entrenaron con el mismo imgsz.
        return json.loads((carpeta / "config.json").read_text())["params"]["receta"]

    def predecir(carpeta, imagenes):
        """Puntaje por imagen y cajas (xc, yc, w, h normalizadas, confianza), como en los notebooks 08 y 09."""
        receta, modelo = receta_de(carpeta), YOLO(str(carpeta / "modelo.pt"))
        puntajes, cajas = [], []
        for inicio in range(0, len(imagenes), 32):
            # Ultralytics espera arrays en BGR, como los lee OpenCV.
            lote = [np.ascontiguousarray(imagen[:, :, ::-1]) for imagen in imagenes[inicio:inicio + 32]]
            for s in modelo.predict(source=lote, imgsz=receta["imgsz"], conf=receta["conf"], iou=receta["iou"],
                                    max_det=receta["max_det"], device=DISPOSITIVO, augment=False, half=False,
                                    save=False, verbose=False):
                conf = s.boxes.conf.cpu().numpy()
                puntajes.append(float(conf.max()) if len(conf) else 0.0)
                cajas.append(np.column_stack([s.boxes.xywhn.cpu().numpy(), conf]))
        return np.array(puntajes), cajas

    def predecir_como_val(carpeta, imagenes, nombres):
        """Cajas para el mAP, con la misma entrada que model.val: imagen agrandada a imgsz y borde gris."""
        receta, modelo = receta_de(carpeta), YOLO(str(carpeta / "modelo.pt"))
        lado = receta["imgsz"]
        forma = math.ceil(lado / 32 + 0.5) * 32  # tamaño del lote en val (stride 32, pad 0.5)
        borde = (forma - lado) // 2
        filas = []
        for nombre, imagen in zip(nombres, imagenes):
            bgr = cv2.resize(np.ascontiguousarray(imagen[:, :, ::-1]), (lado, lado), interpolation=cv2.INTER_LINEAR)
            bgr = cv2.copyMakeBorder(bgr, borde, borde, borde, borde, cv2.BORDER_CONSTANT, value=(114, 114, 114))
            s = modelo.predict(source=bgr, imgsz=forma, conf=receta["conf"], iou=receta["iou"],
                               max_det=receta["max_det"], device=DISPOSITIVO, augment=False, half=False,
                               save=False, verbose=False)[0]
            # Sin recortar a la imagen: model.val compara las cajas en el espacio con borde.
            xyxy = (s.boxes.xyxy.cpu().numpy() - borde) / lado
            filas += [dict(archivo=nombre, confianza=float(c), xc=(x0 + x1) / 2, yc=(y0 + y1) / 2, w=x1 - x0,
                           h=y1 - y0) for (x0, y0, x1, y1), c in zip(xyxy, s.boxes.conf.cpu().numpy())]
        return filas


TOLERANCIA = 0.01  # diferencia máxima aceptada contra predicciones_val.csv; si no, la corrida queda afuera
filas, chequeo, cajas_test, mapas = [], [], {}, []
for _, corrida in corridas.iterrows():
    carpeta = resultados.ruta_corrida(rutas, corrida["modelo"], corrida["modo"])
    print("Prediciendo", corrida["corrida"], "...")
    p_val, _ = predecir(carpeta, imagenes_val)
    guardadas = pd.read_csv(carpeta / "predicciones_val.csv").set_index("archivo").loc[archivos_val, "probabilidad"]
    diferencia = np.abs(p_val - guardadas.to_numpy())
    chequeo.append({"corrida": corrida["corrida"], "diferencia_max": diferencia.max(),
                    "cambian_val": int(((p_val >= corrida["umbral"]) != (guardadas >= corrida["umbral"])).sum())})
    p_test, cajas = predecir(carpeta, imagenes)
    cajas_test[corrida["corrida"]] = cajas
    filas.append(pd.DataFrame({"modelo": corrida["modelo"], "modo": corrida["modo"], "archivo": etiquetas["archivo"],
                               "y_real": y_test, "probabilidad": p_test,
                               "prediccion": (p_test >= corrida["umbral"]).astype(int)}))
    if FAMILIA != "clasificadores":
        mapas.append({"corrida": corrida["corrida"],
                      **deteccion.map_cajas(reales, predecir_como_val(carpeta, imagenes, etiquetas["archivo"]))})
predicciones = pd.concat(filas, ignore_index=True)
chequeo = pd.DataFrame(chequeo)
chequeo["valida"] = chequeo["diferencia_max"] < TOLERANCIA
chequeo.style.format({"diferencia_max": "{:.6f}"})

## 3. Métricas en test

Cada métrica con su intervalo de 95 % por bootstrap (se remuestrean las 200 imágenes, como en el 15) y, al lado,
el valor que la corrida tuvo en val. Con 75 imágenes sin paneles, una sola mueve 1.3 puntos el recall de esa
clase: las diferencias chicas entre modelos no son concluyentes (mirar si los intervalos se superponen).

In [ ]:
invalidas = chequeo.loc[~chequeo["valida"], "corrida"].tolist()
if invalidas:
    print("Quedan afuera porque la carga no reproduce sus predicciones de val (revisar antes de usarlas):", invalidas)
elegidas = [c for c in corridas["corrida"] if c not in invalidas]
valores_val = corridas.set_index("corrida")[[m for m in comparacion.METRICAS if m in corridas]].add_prefix("val_")
tabla = comparacion.intervalos(predicciones, elegidas).join(valores_val, on="corrida")
tabla = tabla.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
columnas = ["corrida"] + [c for m in comparacion.METRICAS for c in (f"val_{m}", m, f"{m}_inf", f"{m}_sup") if c in tabla]
tabla[columnas].style.format({c: "{:.3f}" for c in columnas[1:]}).hide(axis="index")

In [ ]:
if FAMILIA != "clasificadores":
    def map_val(corrida):
        """mAP de val guardado: en metricas.json (extra.deteccion) o, en corridas viejas, metricas_deteccion.json."""
        carpeta = rutas.resultados_equipo / corrida
        metricas = json.loads((carpeta / "metricas.json").read_text())
        guardado = (metricas.get("extra") or {}).get("deteccion")
        if not guardado and (carpeta / "metricas_deteccion.json").is_file():
            guardado = json.loads((carpeta / "metricas_deteccion.json").read_text())
        return (guardado or {}).get("val", {})

    deteccion_test = pd.DataFrame(mapas)
    deteccion_test = deteccion_test[deteccion_test["corrida"].isin(elegidas)].reset_index(drop=True)
    for columna in ("map50", "map50_95"):
        deteccion_test[f"val_{columna}"] = [map_val(c).get(columna) for c in deteccion_test["corrida"]]
    columnas = ["corrida", "val_map50", "map50", "val_map50_95", "map50_95", "precision", "recall"]
    display(deteccion_test[columnas].style.format({c: "{:.3f}" for c in columnas[1:]}, na_rep="—").hide(axis="index"))
    print("precision/recall: de cajas, en el puntaje de F1 máximo (como Ultralytics). El mAP50-95 es muy sensible a",
          "qué tan ajustada está cada caja, y las del test las marcó una sola persona: reportarlo con esa salvedad.")

## 4. Cómo clasificó cada modelo y cómo era realmente

Para la corrida elegida en `CORRIDA` (por defecto, la de mayor balanced accuracy en test) se muestran ejemplos de
las cuatro casillas de la matriz de confusión. **Verde:** cajas anotadas (lo real). **Rojo:** cajas predichas con
confianza mayor o igual al umbral (solo YOLO). El título dice lo real, lo predicho y la probabilidad o puntaje.

Los errores se ordenan por qué tan seguro estaba el modelo: arriba, los más confiados.

In [ ]:
CORRIDA = tabla["corrida"].iloc[0]  # cambiar por cualquier corrida de la tabla
POR_CASILLA = 8

umbral = corridas.set_index("corrida").loc[CORRIDA, "umbral"]
modelo, modo = CORRIDA.split("/")
pred = predicciones[(predicciones["modelo"] == modelo) & (predicciones["modo"] == modo)].reset_index(drop=True)
casillas = {
    "Falsos positivos: dijo CON, era SIN": pred[(pred.y_real == 0) & (pred.prediccion == 1)].sort_values("probabilidad", ascending=False),
    "Falsos negativos: dijo SIN, era CON": pred[(pred.y_real == 1) & (pred.prediccion == 0)].sort_values("probabilidad"),
    "Aciertos CON paneles": pred[(pred.y_real == 1) & (pred.prediccion == 1)].sample(frac=1, random_state=42),
    "Aciertos SIN paneles": pred[(pred.y_real == 0) & (pred.prediccion == 0)].sample(frac=1, random_state=42),
}


def dibujar(ax, i, titulo):
    ax.imshow(imagenes[i], extent=(0, 1, 1, 0))
    for xc, yc, w, h in reales[etiquetas["archivo"][i]]:
        ax.add_patch(Rectangle((xc - w / 2, yc - h / 2), w, h, fill=False, edgecolor="lime", linewidth=1.6))
    if cajas_test[CORRIDA] is not None:
        for xc, yc, w, h, c in cajas_test[CORRIDA][i]:
            if c >= umbral:
                ax.add_patch(Rectangle((xc - w / 2, yc - h / 2), w, h, fill=False, edgecolor="red",
                                       linewidth=1.2, linestyle="--"))
                ax.text(xc - w / 2, yc - h / 2, f"{c:.2f}", color="red", fontsize=7, va="bottom")
    ax.set_title(titulo, fontsize=8)
    ax.set_axis_off()


for nombre, filas_casilla in casillas.items():
    print(f"{nombre}: {len(filas_casilla)} imágenes")
    if filas_casilla.empty:
        continue
    muestra = filas_casilla.head(POR_CASILLA)
    fig, axs = plt.subplots(1, POR_CASILLA, figsize=(2.6 * POR_CASILLA, 3.1), squeeze=False)
    for ax in axs.flat:
        ax.set_axis_off()
    for ax, (i, fila) in zip(axs.flat, muestra.iterrows()):
        real = "CON" if fila.y_real else "SIN"
        predicho = "CON" if fila.prediccion else "SIN"
        dibujar(ax, i, f"{fila.archivo}\nreal {real} · dijo {predicho} · {fila.probabilidad:.2f}")
    fig.suptitle(f"{CORRIDA} · {nombre}", x=0.01, ha="left")
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(pred["y_real"], pred["prediccion"], labels=[0, 1],
                                        display_labels=["sin paneles", "con paneles"], colorbar=False, ax=ax)
ax.set_title(f"{CORRIDA} · test (umbral {umbral})")
plt.tight_layout()
plt.show()

## 5. Imágenes en las que se equivocan casi todos

Para cada imagen se cuenta cuántas corridas de la familia la clasificaron mal. Las que fallan casi todas suelen
ser casos difíciles de verdad (paneles chicos, en el borde o tapados) o dudas de etiquetado: conviene mirarlas
antes de sacar conclusiones, pero **no corregir el test para que den mejor los modelos**.

In [ ]:
MOSTRAR = 12

errores = (predicciones.assign(corrida=predicciones["modelo"] + "/" + predicciones["modo"])
           .query("corrida in @elegidas")
           .assign(error=lambda t: (t["prediccion"] != t["y_real"]).astype(int))
           .groupby("archivo")["error"].sum())
errores = errores.reindex(etiquetas["archivo"])
print(f"Imágenes que fallan todas las {len(elegidas)} corridas: {(errores == len(elegidas)).sum()}")
peores = errores.sort_values(ascending=False).head(MOSTRAR)
peores = peores[peores > 0]
columnas = 6
fig, axs = plt.subplots(math.ceil(len(peores) / columnas) if len(peores) else 1, columnas,
                        figsize=(2.6 * columnas, 3.1 * max(1, math.ceil(len(peores) / columnas))), squeeze=False)
for ax in axs.flat:
    ax.set_axis_off()
for ax, (nombre, n) in zip(axs.flat, peores.items()):
    i = int(np.flatnonzero(etiquetas["archivo"] == nombre)[0])
    ax.imshow(imagenes[i], extent=(0, 1, 1, 0))
    for xc, yc, w, h in reales[nombre]:
        ax.add_patch(Rectangle((xc - w / 2, yc - h / 2), w, h, fill=False, edgecolor="lime", linewidth=1.6))
    ax.set_title(f"{nombre}\nreal {'CON' if y_test[i] else 'SIN'} · fallan {n}/{len(elegidas)}", fontsize=8)
plt.tight_layout()
plt.show()

## 6. Cómo reportarlo

- Decir siempre que es **test** (Kingston, 200 imágenes, etiquetado por una persona) y que los umbrales se
  eligieron en val.
- Reportar cada métrica con su intervalo. Una diferencia entre dos corridas es clara solo si los intervalos no
  se superponen; para afirmarlo bien, usar `comparacion.diferencias(predicciones, elegidas, referencia)`, que la
  mide pareada.
- En detección, el **mAP50** es la métrica principal; el **mAP50-95** depende mucho de cómo se dibujó cada caja.
- Si se entrena un modelo nuevo, vuelve a entrar acá solo cuando sea vigente: no ajustar nada mirando el test.